# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [ ]:
print('Ready. No external packages required for this practical.')

## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [ ]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [ ]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [ ]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [ ]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [ ]:
# TODO: your extended RULES table and a test of each new rule

RULES = [
    # --- Original procedure rules ---
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),

    # --- New Rule 1 (captures phrase) ---
    (['i', 'feel', '?X'],    'Why do you feel {X}?'),

    # --- New Rule 2 (captures phrase) ---
    (['i', 'want', '?X'],    'What would it mean to you to get {X}?'),

    # --- New Rule 3 ---
    (['i', 'like', '?X'],    'What do you like most about {X}?'),

    # --- New Rule 4 ---
    (['i', 'fear', '?X'],    'Can you tell me more about your fear of {X}?'),

    # --- New Rule 5 (two fixed words + captured phrase) ---
    (['i', 'am', 'afraid', 'of', '?X'],
                             'What specifically about {X} frightens you?'),
]

FALLBACK = 'Tell me more about that.'

def swap_person(tokens):
    SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}
    return [SWAP.get(t, t) for t in tokens]

def match(pattern, tokens):
    """Basic tail-capture matcher (same as procedure)."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None
    return bindings if len(pattern) == len(tokens) else None

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

# Test every new rule
tests = [
    'I am worried about my exams',   # original
    'I need more time',              # original
    'I cannot sleep',                # original
    'I feel anxious',                # new 1
    'I want a holiday',              # new 2
    'I like my new job',             # new 3
    'I fear failure',                
    'I am afraid of spiders',        
]
for s in tests:
    print('%-32s -> %s' % (s, respond(s)))

### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [ ]:
# TODO: an improved matcher, plus tests showing it works

def match_advanced(pattern, tokens):
    """Return a dict of bindings if pattern fits tokens, else None.
    Supports a single '?X'-style variable at any position."""
    var_pos = None
    var_name = None
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            var_pos, var_name = i, p
            break

    # No variable: pattern must equal the tokens exactly
    if var_pos is None:
        return {} if pattern == tokens else None

    prefix = pattern[:var_pos]
    suffix = pattern[var_pos + 1:]

    # Prefix must match the start of tokens
    if tokens[:len(prefix)] != prefix:
        return None
    # Suffix must match the end of tokens
    if len(tokens) < len(prefix) + len(suffix):
        return None
    if suffix and tokens[-len(suffix):] != suffix:
        return None

    # Capture the middle
    middle = tokens[len(prefix): len(tokens) - len(suffix)]
    return {var_name: middle}


# ---------------- Tests ----------------
tests = [
    (['i', '?X', 'about', 'exams'], 'i am worried about exams'),
    (['i', '?X', 'about', 'exams'], 'i am very worried about exams'),
    (['i', '?X', 'about', 'exams'], 'i am worried about my exams'),
    (['i', 'am', '?X'],             'i am worried about my exams'),
    (['i', '?X'],                   'i feel great today'),
    (['i', '?X', 'about', 'exams'], 'she is worried about exams'),
]
for pat, sent in tests:
    print(pat, '<=', sent, '->', match_advanced(pat, sent.split()))

### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [ ]:
# TODO: represent expressions as nested lists and apply rewrite rules
expr = ['+', ['*', 1, 'x'], 0]
print('start:', expr)


def simplify_once(e):
    """Apply ONE rewrite pass, recursively, over the whole tree."""
    if not isinstance(e, list):
        return e

    op = e[0]
    args = [simplify_once(a) for a in e[1:]]

    # ---- Rewrite rules ----
    if op == '*':
        a, b = args
        if a == 1: return b          
        if b == 1: return a         
        if a == 0 or b == 0: return 0  
    elif op == '+':
        a, b = args
        if a == 0: return b
        if b == 0: return a

    return [op] + args


def simplify_fully(e):
    """Apply simplify_once until nothing changes."""
    while True:
        new_e = simplify_once(e)
        if new_e == e:
            return e
        e = new_e


print('simplified:', simplify_fully(expr))

# Extra tests
more = [
    ['+', 0, ['*', 'y', 1]],
    ['*', 0, 'z'],
    ['+', ['*', 1, 'a'], ['*', 'b', 1]],
]
for m in more:
    print(m, '->', simplify_fully(m))

### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [ ]:
# TODO: no code needed - record your findings in the observations table

print("Type a sentence (empty line to quit).")
while True:
    s = input("you > ").strip()
    if not s:
        break
    print("bot >", respond(s))

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

In [ ]:
# TODO: write your comparison in a markdown cell below this one
### What the program *actually* stores

- A list of `RULES`, each a pair: `(pattern_tokens, reply_template_string)`.
- A dictionary `SWAP` mapping first-person words → second-person words.
- A `FALLBACK` string.
- At runtime, a short-lived `bindings` dict per input, mapping the variable
  name (e.g. `'?X'`) to a **list of string tokens**.
- In Task 3: nested Python lists representing arithmetic expressions, and a
  set of hard-coded rewrite rules for `+` and `*`.

### What a user typically *assumes* it stores

- The meaning of words like *worried*, *exams*, *afraid*, *need*.
- A memory of earlier turns in the conversation.
- A model of the user as a person with feelings, goals, and history.
- An intent behind each question (e.g. that "Why do you feel…" is real curiosity).

### The gap

The program contains **no semantics, no memory, no model of the world, and no
notion of "meaning"**. It compares only the *shape* of the token list against
fixed templates. Any apparent understanding is supplied entirely by the human
reader, who interprets the output as if it were addressed to them.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| I am worried about my exams | `['i','am','?X']` | How long have you been worried about your exams? | Yes |
| I need more time | `['i','need','?X']` | Why do you need more time? | Yes |
| I feel anxious | `['i','feel','?X']` | Why do you feel anxious? | Yes |
| The weather is fine today | none (fallback) | Tell me more about that. | Partly |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| Understood the word "exams" and what stress about them means | A list of tokens and a template string |
| Remembered the earlier mention of exams | Nothing between turns — no state |
| Was genuinely curious about their feelings | A `format()` call filling a fixed template |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

*Your answers:*

1. It understands nothing. It stores token lists, template strings, and a
   word-swap dictionary. Every "sensible" reply is produced by comparing the
   *shape* of the input against a pattern and dropping the captured tokens
   into a pre-written sentence. The meaning is projected by the human reader,
   not represented in the program.

2. Swapping 'my'→'your' and 'i'→'you' makes the reply read as if the program
   has taken the user's perspective and is asking a follow-up. Without the
   swap the reply sounds like an echo; with it, the same captured tokens
   become a question addressed back to the user. The change is purely
   syntactic, but it exploits the human habit of reading turn-taking as
   engagement.

3. If fluent output can convince even people who know the mechanism, then
   subjective impressions are a poor metric for evaluating conversational
   systems. Evaluating modern AI requires external, objective tests of
   reasoning, factual accuracy, and consistency — not just how "understood"
   a user feels.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [6]:
NAME = "Sajawal"
ROLL_NUMBER = "2025-cse-032"
print(NAME, ROLL_NUMBER)

Sajawal 2025-cse-032
